In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

100%|██████████| 170M/170M [00:09<00:00, 18.2MB/s]


In [4]:
image, label = train_data[0]

In [5]:
image

tensor([[[-0.5373, -0.6627, -0.6078,  ...,  0.2392,  0.1922,  0.1608],
         [-0.8745, -1.0000, -0.8588,  ..., -0.0353, -0.0667, -0.0431],
         [-0.8039, -0.8745, -0.6157,  ..., -0.0745, -0.0588, -0.1451],
         ...,
         [ 0.6314,  0.5765,  0.5529,  ...,  0.2549, -0.5608, -0.5843],
         [ 0.4118,  0.3569,  0.4588,  ...,  0.4431, -0.2392, -0.3490],
         [ 0.3882,  0.3176,  0.4039,  ...,  0.6941,  0.1843, -0.0353]],

        [[-0.5137, -0.6392, -0.6235,  ...,  0.0353, -0.0196, -0.0275],
         [-0.8431, -1.0000, -0.9373,  ..., -0.3098, -0.3490, -0.3176],
         [-0.8118, -0.9451, -0.7882,  ..., -0.3412, -0.3412, -0.4275],
         ...,
         [ 0.3333,  0.2000,  0.2627,  ...,  0.0431, -0.7569, -0.7333],
         [ 0.0902, -0.0353,  0.1294,  ...,  0.1608, -0.5137, -0.5843],
         [ 0.1294,  0.0118,  0.1137,  ...,  0.4431, -0.0745, -0.2784]],

        [[-0.5059, -0.6471, -0.6627,  ..., -0.1529, -0.2000, -0.1922],
         [-0.8431, -1.0000, -1.0000,  ..., -0

In [6]:
label

6

In [7]:
image.shape

torch.Size([3, 32, 32])

In [8]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [9]:
class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5)  # (input size - kernal size ==> 32 - 5 ==> 27 / stride ==> 27 / 1 (default stride is 1) ==> 27+1 ==> so the new channel with (12, 28, 28) means 12 channel with 28*28 pixels)
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14)
        self.conv2 = nn.Conv2d(12, 24, 5) #(14-5 ==> 9 / stride ==> 9/1 ==> 9+1 ==> 10 ==> (24, 10, 10)) ==> pool will apply ==> (24, 5, 5) ==> flatten will apply ==> (24 * 5 * 5)
        self.fc1 = nn.Linear((24 * 5 * 5), 120) # we can play with out feature. instead of 120 we can give 140 whatever we need
        self.fc2 = nn.Linear(120, 84) # we can play with out feature. instead of 84 we can give 92 or anything
        self.fc3 = nn.Linear(84, 10) # hear out feature should be 10 neurans because this is probability of 10 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [10]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [11]:
for epoch in range(15):
    print(f'Training epoch {epoch}...🔥')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss = running_loss + loss.item()

    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training epoch 0...🔥
Loss: 2.1999
Training epoch 1...🔥
Loss: 1.7587
Training epoch 2...🔥
Loss: 1.5359
Training epoch 3...🔥
Loss: 1.4198
Training epoch 4...🔥
Loss: 1.3258
Training epoch 5...🔥
Loss: 1.2491
Training epoch 6...🔥
Loss: 1.1784
Training epoch 7...🔥
Loss: 1.1164
Training epoch 8...🔥
Loss: 1.0638
Training epoch 9...🔥
Loss: 1.0210
Training epoch 10...🔥
Loss: 0.9794
Training epoch 11...🔥
Loss: 0.9416
Training epoch 12...🔥
Loss: 0.9076
Training epoch 13...🔥
Loss: 0.8789
Training epoch 14...🔥
Loss: 0.8435


In [12]:
torch.save(net.state_dict(), 'trained_net.pth')

In [13]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [14]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')

Accuracy: 66.01%


In [16]:
new_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(img_path):
    image = Image.open(img_path)
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

img_paths = ['/content/test/d.jpg', '/content/test/f.jpg', '/content/test/p.jpg']

images = [load_image(img) for img in img_paths]

net.eval()
with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction: {class_names[predicted.item()]}')


Prediction: horse
Prediction: frog
Prediction: plane
